                # Advanced Circuit Hierarchy

                This notebook documents the hierarchical circuit APIs used to split a multi-mode circuit into diagonalized subsystems with symbolic interaction terms.
                

                **Audience:** readers already comfortable with the basic `Circuit` workflow.

                **Prerequisites:** symbolic circuit decomposition, subsystem truncation, and the `HilbertSpace` concepts from notebook 05.

                **Learning goals:** build flat and nested hierarchy specifications, generate truncation templates, use `hierarchical_diag` directly, and use `configure!` to cache subsystem decompositions and symbolic interaction terms on a circuit.
                

                ## Outline

                1. Build typed hierarchy objects with `HierarchyLeaf` and `HierarchyGroup`.
                2. Generate scqubits-style truncation templates.
                3. Compare direct `hierarchical_diag` against `configure!`.
                4. Inspect `SubCircuit` objects and symbolic subsystem interactions.
                

In [ ]:
                using ScQubitsMimic

                typed_hierarchy = HierarchyGroup([
                    HierarchyGroup([HierarchyLeaf([1]), HierarchyLeaf([2])]),
                    HierarchyLeaf([3]),
                ])

                (
                    typed_hierarchy isa HierarchyNode,
                    truncation_template(typed_hierarchy),
                )
                

In [ ]:
                desc_two_mode = """
                branches:
                  - [JJ, 0, 1, EJ=10.0, EC=0.3]
                  - [JJ, 0, 2, EJ=8.0, EC=0.4]
                  - [C, 1, 2, EC=0.1]
                """

                circ_two_mode = Circuit(desc_two_mode; ncut=8)
                hs_hd = hierarchical_diag(circ_two_mode;
                    system_hierarchy=[[1], [2]],
                    subsystem_trunc_dims=[8, 8],
                )

                (
                    subsystem_types = [typeof(sub) for sub in hs_hd.subsystems],
                    subsystem_dims = [hilbertdim(sub) for sub in hs_hd.subsystems],
                    dressed_evals = round.(eigenvals(hs_hd; evals_count=6), digits=6),
                )
                

In [ ]:
                configure!(circ_two_mode; system_hierarchy=[[1], [2]], subsystem_trunc_dims=[8, 8])
                (
                    hierarchical_flag = circ_two_mode._hierarchical_diagonalization,
                    stored_subsystems = [typeof(sub) for sub in circ_two_mode._subsystems],
                    subsystem_hamiltonian_1 = sym_hamiltonian(circ_two_mode; subsystem_index=1, return_expr=true),
                    subsystem_hamiltonian_2 = sym_hamiltonian(circ_two_mode; subsystem_index=2, return_expr=true),
                    interaction_12 = sym_interaction(circ_two_mode; subsystem_indices=(1, 2), return_expr=true),
                )
                

In [ ]:
                desc_nested = """
                branches:
                  - [JJ, 0, 1, EJ=10.0, EC=0.3]
                  - [JJ, 0, 2, EJ=8.0, EC=0.4]
                  - [JJ, 0, 3, EJ=6.0, EC=0.5]
                  - [C, 1, 2, EC=0.08]
                  - [C, 2, 3, EC=0.06]
                """

                circ_nested = Circuit(desc_nested; ncut=5)
                nested_hierarchy = [[[1], [2]], [3]]
                nested_truncation = truncation_template(nested_hierarchy; individual_trunc_dim=5, combined_trunc_dim=20)
                configure!(circ_nested; system_hierarchy=nested_hierarchy, subsystem_trunc_dims=nested_truncation)

                (
                    nested_truncation = nested_truncation,
                    top_level_subsystems = [typeof(sub) for sub in circ_nested._subsystems],
                    top_level_dims = [hilbertdim(sub) for sub in circ_nested._subsystems],
                    interaction_12 = sym_interaction(circ_nested; subsystem_indices=(1, 2), return_expr=true),
                )
                

                ## Exercise

                Starting from the two-mode circuit, change the hierarchy to a single grouped subsystem `[[1, 2]]` and compare the returned Hilbert-space dimension with the two-subsystem decomposition.
                

In [ ]:
                circ_exercise = Circuit(desc_two_mode; ncut=8)
                hs_single_group = hierarchical_diag(circ_exercise;
                    system_hierarchy=[[1, 2]],
                    subsystem_trunc_dims=[12],
                )
                [hilbertdim(sub) for sub in hs_single_group.subsystems]
                

                ## Pitfalls And Extensions

                `configure!` is strict about `subsystem_trunc_dims`: if you omit it, the call fails by design. Use `truncation_template(system_hierarchy)` as the starting point and then tighten the truncation manually.

                `SubCircuit` objects are produced by the hierarchy workflow; they are not meant to be constructed directly. Use `hierarchical_diag` for one-shot workflows and `configure!` when you want the circuit to keep cached subsystem state and symbolic decompositions.
                

                ## API Covered

                `HierarchyLeaf`, `HierarchyGroup`, `HierarchyNode`, `SubCircuit`, `truncation_template`, `hierarchical_diag`, `configure!`, and `sym_interaction`.
                